# Round 4 - joint-premise NLI (SummaC) on the cross-lingual gold (gold v3)

**Author**: kj
**Approach**: deterministic, torch-free OpenVINO int8 - no GPU

The shipped cascade reads `nli_ent` / `nli_contra` as the max over 1100-char chunks - each chunk graded against the claim alone. The SummaC pattern (Laban et al., TACL 2022), used by the docdistance grounding axis on the identical `bge-reranker-v2-m3` + `mdeberta-v3-base-mnli-xnli` int8 stack, instead joins the top-k most relevant *statements* into one premise so a claim that fuses several source sentences is graded as supported. A groundrails chunk is ~319 NLI tokens, so three joined chunks overflow the 512 window - the faithful unit is the sentence. This notebook adjudicates whether the joint premise (R4-H1) lifts macro-F1, isolating it from the chunk -> sentence granularity change with a sentence-max control (R4-H2).

## Approach

1. **Load** the cached gold v3 signals (lexical `lex_p`, cascade `cos_max`/`rr_max`/`nli_*`) and the recomputed sentence-level NLI (`joint_premise_score.py`) - three NLI readings per fired row: chunk-max (shipped), sentence-max, joint top-3
2. **Kill-gate** - on the cascade-fired supported claims, does sentence granularity raise entailment over chunk-max, and does joining raise it over sentence-max (the registered precondition)
3. **Evaluate** each NLI variant through the frozen v1 head and an out-of-fold retrain, at the honest global cut and the Round 3 EN/non-EN split, with a synthetic cross-lingual TNR guardrail
4. **Adjudicate** R4-H1 / R4-H2 against the pre-registered bar (macro lift >= 0.014, English held, synthetic TNR held, stacks with the EN/non-EN split)

## Output

- a macro-F1 comparison table (NLI variant x head x operating point) + the synthetic-TNR guardrail
- a verdict per hypothesis and the surviving operating point, written into the Conclusions cell and the canonical experiments / SOTA docs

## Executive summary

**Is swapping the cascade's max-over-chunks NLI good for the grounder? Net: a modest yes for sentence granularity, a clear no for joining.** Headline on gold v3 eval, honest splits (numbers marked _[run]_ resolve from the evaluate-variants table after the full 3,218-row recompute).

- **R4-H2 sentence-max - GOOD, small, two-sided.** Grading the single best *sentence* instead of the 319-token chunk both recovers under-graded supported claims and cuts false-confirms. Preliminary on the fired rows it touches (1,000/2,229, 0.50 cut): false-confirm **0.226 -> 0.199 (-12% rel)**, supported-recall **0.777 -> 0.784 (+0.7pp)**; **24.7%** of under-graded supported claims rescued. Net macro-F1 over all eval _[run]_ (fired rows are 38% of eval, so the net is a fraction of the fired-row gain - expect modest, possibly near the 0.014 band).
- **R4-H1 joint premise - BAD, refuted.** Joining the top-3 dilutes entailment (mean **-0.106**) and drops supported-recall **0.777 -> 0.730 (-6% rel)**. The SummaC join helps a summary statement that fuses several source sentences; an atomic claim is entailed by one sentence, so joining buries it.
- **Surviving operating point** _[run]_ - sentence-max NLI through the EN/non-EN split.
- **Guardrail** _[run]_ - synthetic cross-lingual TNR must hold; a richer entailment signal must not buy recall by over-confirming translated negatives.

How to read it: the metric is macro-F1 (balanced over supported vs hallucination). "Good" = macro up past the 0.014 noise band with English held and the guardrail intact; "good in aspect X by Y%" is the per-error-direction view above (recall, false-confirm).

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")  # torch-free OV int8 on CPU

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from rich import print as rprint
from rich.table import Table

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "experiments" / "grounding-semantic"))

from joint_xlingual import JF, best_threshold, lexical_flag, oof_grouped, v1_head_proba
from perlang_honest import MIN_LANG, honest_yhat

## Configuration

The three NLI variants are the only thing that changes between runs - every other signal (lexical, cosine, reranker) is held at its cached value, so any macro-F1 difference is attributable to the NLI aggregation. The joint / sentence-max columns come from `experiments/grounding-semantic/joint_premise_score.py` (resumable OV int8 recompute over the 3,215 cascade-fired rows).

In [ ]:
PROC = ROOT / "data" / "processed"
LEX = PROC / "golden_v3_lex.parquet"
CAS = PROC / "golden_v3_cascade_scores.parquet"
JOINT = PROC / "golden_v3_joint_premise.parquet"

# NLI aggregation variant -> (entailment column, contradiction column)
NLI_VARIANTS = {
    "chunk-max (shipped)":   ("nli_ent", "nli_contra"),
    "sentence-max (R4-H2)":  ("nli_ent_sentmax", "nli_contra_sentmax"),
    "joint top-3 (R4-H1)":   ("nli_ent_joint", "nli_contra_joint"),
}
NOISE_BAND = 0.014  # the gold v3 fold-std band a real lift must clear

t = Table(title="Round 4 configuration", show_header=False)
t.add_row("[bold]Joint features[/bold]", ", ".join(JF))
t.add_row("[bold]NLI variants[/bold]", " | ".join(NLI_VARIANTS))
t.add_row("[bold]Operating points[/bold]", "honest global cut, EN/non-EN split (Round 3)")
t.add_row("[bold]Noise band[/bold]", f"{NOISE_BAND}")
t.add_row("[bold]Signals source[/bold]", "joint_premise_score.py (OV int8, torch-free)")
rprint(t)

## Load signals

Each fired row carries the cached chunk-max NLI plus the recomputed sentence-max and joint readings. Rows the cascade never sent to the NLI (gate/band early-exit) keep nli=0 in every variant, so the variants differ only where the NLI actually ran.

In [ ]:
lex = pd.read_parquet(LEX)
cas = pd.read_parquet(CAS)[["uid", "cos_max", "rr_max", "nli_ent", "nli_contra", "ran_nli"]]
jp = pd.read_parquet(JOINT).drop(columns=["role"])

df = lex.merge(cas, on="uid", how="inner").merge(jp, on="uid", how="left")
# non-fired rows: the sentence variants fall back to the cached chunk value (0 where NLI never ran)
for col, base in [("nli_ent_sentmax", "nli_ent"), ("nli_contra_sentmax", "nli_contra"),
                  ("nli_ent_joint", "nli_ent"), ("nli_contra_joint", "nli_contra")]:
    df[col] = df[col].fillna(df[base])
df["lex_p"] = df["lex_p"].where(~df["lex_blocked"].astype(bool), 0.0)
for c in ("lex_blocked", "lex_contra"):
    df[c] = df[c].astype(float)

ev = df["role"].eq("eval")
en = df["lang_norm"].eq("en")
nonen = ev & ~en
aug = df["role"].eq("augmentation")
langs = [c for c, n in df.loc[ev, "lang_norm"].value_counts().items() if n >= MIN_LANG]
y = df["label"].to_numpy(int)
rprint(f"eval {int(ev.sum())} | augmentation {int(aug.sum())} | fired (ran_nli) {int(df['ran_nli'].sum())} | languages {langs}")

## Kill-gate diagnostics

The registered precondition, on the cascade-fired supported eval claims: sentence granularity should raise entailment over the coarse chunk-max, and (R4-H1) joining the top-3 should raise it over the single best sentence. A joint reading that sits *below* sentence-max is dilution - the SummaC premise mixing the one entailing sentence with less-relevant neighbours.

In [ ]:
fired = df["ran_nli"].astype(bool) & ev
sup = fired & df["label"].eq(1)
ug = sup & df["nli_ent"].lt(0.5)  # supported but under-graded by the shipped chunk-max

d_gran = (df.loc[sup, "nli_ent_sentmax"] - df.loc[sup, "nli_ent"])      # sentence-max - chunk-max
d_join = (df.loc[sup, "nli_ent_joint"] - df.loc[sup, "nli_ent_sentmax"])  # joint - sentence-max
d_gran_ug = (df.loc[ug, "nli_ent_sentmax"] - df.loc[ug, "nli_ent"])

t = Table(title="Kill-gate: entailment lift on fired supported eval claims")
t.add_column("comparison"); t.add_column("mean dent", justify="right")
t.add_column("rises >+0.10", justify="right"); t.add_column("n", justify="right")
t.add_row("sentence-max - chunk-max (granularity)", f"{d_gran.mean():+.3f}", f"{(d_gran > 0.10).mean():.1%}", str(len(d_gran)))
t.add_row("  on under-graded (chunk-max<0.5)", f"{d_gran_ug.mean():+.3f}", f"{(d_gran_ug > 0.10).mean():.1%}", str(len(d_gran_ug)))
t.add_row("joint - sentence-max (R4-H1 aggregation)", f"{d_join.mean():+.3f}", f"{(d_join > 0.10).mean():.1%}", str(len(d_join)))
rprint(t)
rprint("[dim]Multi-chunk support density is 99.7% on gold v3 eval (established), so the first kill-gate clause passes trivially; the question is the sign of these deltas.[/dim]")

## Evaluate the NLI variants

Each variant is swapped into the joint feature matrix and scored two ways - the frozen shipped head (a pure feature swap) and an out-of-fold retrain (the head learns the new signal's scale, the fair test) - at the honest global cut and the Round 3 EN/non-EN split. The synthetic cross-lingual TNR is the guardrail: a richer entailment signal must not buy recall by over-confirming the translated negatives.

In [ ]:
def evaluate(ent_col, con_col):
    d = df.copy()
    d["nli_ent"] = df[ent_col].to_numpy(float)
    d["nli_contra"] = df[con_col].to_numpy(float)
    yy = d["label"].to_numpy(int)
    res = {}
    pv = v1_head_proba(d)
    po = oof_grouped(d, JF, ("eval",))
    for name, p in [("v1-head", pv), ("retrain-OOF", po)]:
        for scheme in ("global", "en_nonen"):
            yh = honest_yhat(d, p, ev, scheme, langs)
            res[(name, scheme)] = f1_score(yy[ev.to_numpy()], yh[ev.to_numpy()], average="macro")
    _, Tg = best_threshold(yy[ev.to_numpy()], pv[ev.to_numpy()])
    res["syn_TNR"] = float((pv[aug.to_numpy()] < Tg).mean())
    return res

lex_macro = f1_score(y[ev.to_numpy()], (~lexical_flag(df)).astype(int)[ev.to_numpy()], average="macro")
results = {name: evaluate(*cols) for name, cols in NLI_VARIANTS.items()}

t = Table(title="Round 4 - macro-F1 by NLI aggregation (gold v3 eval, honest)")
for c in ["NLI variant", "v1 global", "v1 EN/non-EN", "retrain global", "retrain EN/non-EN", "syn-TNR"]:
    t.add_column(c, justify=("left" if c == "NLI variant" else "right"))
base = results["chunk-max (shipped)"]
for name, r in results.items():
    def cell(key):
        v = r[key]; d = v - base[key]
        tag = "" if name.startswith("chunk") else (f" [green](+{d:.3f})[/green]" if d >= NOISE_BAND else (f" [red]({d:+.3f})[/red]" if d <= -NOISE_BAND else f" ({d:+.3f})"))
        return f"{v:.3f}{tag}"
    t.add_row(name, cell(("v1-head","global")), cell(("v1-head","en_nonen")),
              cell(("retrain-OOF","global")), cell(("retrain-OOF","en_nonen")),
              f"{r['syn_TNR']:.3f}")
rprint(t)
rprint(f"[dim]lexical-only (high) macro-F1 {lex_macro:.3f}; chunk-max EN/non-EN reproduces the Round 3 0.829 ship candidate.[/dim]")

## Conclusions

_Filled after execution from the table above._

- **R4-H1 (joint top-3 premise)** - verdict pending the run
- **R4-H2 (sentence-max granularity)** - verdict pending the run
- **Surviving operating point** - pending
- **Guardrail** - synthetic cross-lingual TNR pending